# Extracting GPS Coordinates from Google Maps Share LinksField teams collect locations by tapping **Share** in the Google Maps app, whichproduces a short link like `https://maps.app.goo.gl/XXXXXXXX`. That link is uselessfor analysis on its own - you cannot plot it, measure a distance from it or join itto anything.This notebook turns a column of those links into a clean latitude/longitude table.

## Why this needs a browserA short Maps link cannot be expanded with a plain HTTP request. Google answers witha JavaScript interstitial and, in most regions, a cookie consent page. So we drive aheadless Chrome session, let it follow the redirect, and read the coordinates out ofthe **final** URL.That final URL carries the position twice, and the two are not the same:![Anatomy of a resolved Google Maps URL](../docs/images/url-anatomy.png)| Fragment | What it is | Precision ||---|---|---|| `!3d<lat>!4d<lng>` | the **map pin** - the place itself | exact || `@<lat>,<lng>,17z` | the **camera** - where the viewport is centred | a few metres off |Reading `@` is the common mistake. This notebook always prefers `!3d`/`!4d` andfalls back through progressively weaker patterns, recording which one was usedin a `Source` column so low-confidence rows stay visible.

## 1. Install dependenciesOn Google Colab, run this once per session.

In [ ]:
# Chrome + driver for headless browsing (Colab / Debian-based runtimes)
!apt-get update -qq
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y -qq ./google-chrome-stable_current_amd64.deb

!pip install -q selenium xlsxwriter openpyxl

In [ ]:
import os
import re
import time
from urllib.parse import urlparse, parse_qs, unquote

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait

## 2. ConfigurationEvery path lives here. Nothing about a specific dataset is hardcoded further down,so this notebook can be pointed at a different file without editing any logic.> **Note on data.** The repository ships a synthetic dataset of public Indonesian> tourist destinations. Point `INPUT_FILE` at your own workbook to use it for real,> and keep that file out of version control - see `.gitignore`.

In [ ]:
# --- Input -----------------------------------------------------------------
INPUT_FILE  = os.getenv("GMAPS_INPUT_FILE", "../data/sample_tourist_places.xlsx")
INPUT_SHEET = os.getenv("GMAPS_INPUT_SHEET", "PLACES")
LINK_COLUMN = os.getenv("GMAPS_LINK_COLUMN", "LINK GMAPS")

# --- Output ----------------------------------------------------------------
OUTPUT_FILE  = os.getenv("GMAPS_OUTPUT_FILE", "../output/gps_coordinates.xlsx")
OUTPUT_SHEET = "GPS"

# --- Scraping behaviour ----------------------------------------------------
PAGE_TIMEOUT   = 20    # seconds to wait for the redirect to settle
SETTLE_SECONDS = 2     # extra pause after the URL stops changing
ROW_DELAY      = 0     # seconds between rows; raise this for large batches

print(f"Input : {INPUT_FILE} (sheet {INPUT_SHEET!r}, column {LINK_COLUMN!r})")
print(f"Output: {OUTPUT_FILE}")

## 3. Load the dataset

In [ ]:
df_places = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET)

print(f"{len(df_places)} rows loaded")
df_places.head()

## 4. The extraction functionsThree pieces:- `clean_coord` - validates that a captured string really is a decimal degree- `make_driver` - builds one reusable Chrome session with the consent cookie injected- `get_accurate_coords` - resolves a link and walks the fallback chainThe consent cookie is injected **once at startup** rather than per request, and thedriver is reused across the whole batch. Launching Chrome per row is what turns afive-minute job into a two-hour one.

In [ ]:
CONSENT_COOKIE = {
    "name": "SOCS",
    # Generic, account-independent value: it only records that the consent
    # dialog has been answered. No personal data.
    "value": "CAESEwgDEgk0ODE3Nzk3MjMaAmVuIAEaBgiA_LyaBg",
    "domain": ".google.com",
    "path": "/",
    "secure": True,
    "httpOnly": False,
}

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def clean_coord(coord_str):
    """Return the value if it is a bare decimal degree, otherwise None."""
    if not coord_str:
        return None
    candidate = str(coord_str).strip()
    return candidate if re.match(r"^-?\d{1,3}\.\d+$", candidate) else None


def make_driver():
    """Create a headless Chrome driver with the consent cookie pre-injected."""
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument(f"--user-agent={USER_AGENT}")

    driver = webdriver.Chrome(options=chrome_options)

    # Visit google.com once so the cookie has a domain to attach to.
    driver.get("https://www.google.com")
    time.sleep(1)
    driver.add_cookie(CONSENT_COOKIE)
    return driver

In [ ]:
def get_accurate_coords(url, driver=None):
    """Resolve one Google Maps link to {Latitude, Longitude, Source}.

    Strategies run in order of trustworthiness; the first hit wins and its name
    is returned in `Source` so weaker matches remain auditable.
    """
    empty = {"Latitude": None, "Longitude": None, "Source": None}

    if not url or not isinstance(url, str) or not url.strip().startswith("http"):
        return empty

    owns_driver = driver is None
    if owns_driver:
        driver = make_driver()

    try:
        # --- 1. Follow the redirect -------------------------------------
        driver.get(url.strip())
        try:
            WebDriverWait(driver, PAGE_TIMEOUT).until(
                lambda d: "!3d" in d.current_url or "consent.google.com" in d.current_url
            )
            time.sleep(SETTLE_SECONDS)
        except Exception:
            pass  # a timeout is not fatal - parse whatever we have

        # --- 2. Clear the consent interstitial if it appears -------------
        if "consent.google.com" in driver.current_url:
            params = parse_qs(urlparse(driver.current_url).query)
            if "continue" in params:
                destination = unquote(params["continue"][0])
                driver.get("https://www.google.com")
                time.sleep(1)
                driver.add_cookie(CONSENT_COOKIE)
                driver.get(destination)
                try:
                    WebDriverWait(driver, PAGE_TIMEOUT).until(
                        lambda d: "!3d" in d.current_url or "@" in d.current_url
                    )
                    time.sleep(SETTLE_SECONDS)
                except Exception:
                    pass

        final_url = driver.current_url
        html = driver.page_source

        # --- 3. Map pin in the URL (exact) ------------------------------
        lat = re.search(r"!3d(-?\d{1,3}\.\d+)", final_url)
        lng = re.search(r"!4d(-?\d{1,3}\.\d+)", final_url)
        if lat and lng:
            return {"Latitude": clean_coord(lat.group(1)),
                    "Longitude": clean_coord(lng.group(1)),
                    "Source": "pin_url"}

        # --- 4. Map pin in the page source (exact) ----------------------
        pin = re.search(r"!3d(-?\d{1,3}\.\d+)!4d(-?\d{1,3}\.\d+)", html)
        if pin:
            return {"Latitude": clean_coord(pin.group(1)),
                    "Longitude": clean_coord(pin.group(2)),
                    "Source": "pin_html"}

        # --- 5. Camera centre (approximate) -----------------------------
        camera = re.search(r"@(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)", final_url)
        if camera:
            return {"Latitude": clean_coord(camera.group(1)),
                    "Longitude": clean_coord(camera.group(2)),
                    "Source": "camera_url"}

        # --- 6. Coordinates passed as a query parameter -----------------
        params = parse_qs(urlparse(final_url).query)
        for key in ("q", "query", "ll"):
            if key in params:
                match = re.match(r"(-?\d{1,3}\.\d+),\s*(-?\d{1,3}\.\d+)", params[key][0])
                if match:
                    return {"Latitude": clean_coord(match.group(1)),
                            "Longitude": clean_coord(match.group(2)),
                            "Source": "query_param"}

        # --- 7. Any high-precision pair left in the URL -----------------
        loose = re.search(r"(-?\d{1,3}\.\d{4,}),(-?\d{1,3}\.\d{4,})", final_url)
        if loose:
            return {"Latitude": clean_coord(loose.group(1)),
                    "Longitude": clean_coord(loose.group(2)),
                    "Source": "loose_url"}

        # --- 8. Coordinates in the inlined JS payload -------------------
        js = re.search(r"\[null,null,(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)\]", html)
        if js:
            return {"Latitude": clean_coord(js.group(1)),
                    "Longitude": clean_coord(js.group(2)),
                    "Source": "js_html"}

    except Exception as exc:
        print(f"  ! could not resolve {url[:60]}: {exc}")

    finally:
        if owns_driver:
            driver.quit()

    return empty

## 5. Quick sanity checkTwo public landmarks, so the check can be re-run by anyone without touchingprivate data.

In [ ]:
TEST_LINKS = {
    "Candi Borobudur": (
        "https://www.google.com/maps/place/Candi+Borobudur"
        "/@-7.6078574,110.2038233,17z/data=!3m1!4b1!4m6!3m5!1s0x0:0x0"
        "!8m2!3d-7.6078738!4d110.2037342"
    ),
    "Monumen Nasional": (
        "https://www.google.com/maps/place/Monumen+Nasional"
        "/@-6.1753760,106.8272419,17z/data=!3m1!4b1!4m6!3m5!1s0x0:0x0"
        "!8m2!3d-6.1753924!4d106.8271528"
    ),
}

driver = make_driver()
try:
    for name, link in TEST_LINKS.items():
        result = get_accurate_coords(link, driver=driver)
        print(f"{name:20s} -> {result['Latitude']}, {result['Longitude']}  ({result['Source']})")
finally:
    driver.quit()

> Replace the values above with your own `https://maps.app.goo.gl/...` links to test> the short-link path end to end.

## 6. Run the whole datasetOne driver for the entire batch, closed in a `finally` so a crash mid-run neverleaves an orphaned Chrome process behind.

In [ ]:
driver = make_driver()
results = []

try:
    total = len(df_places)
    for position, url in enumerate(df_places[LINK_COLUMN], start=1):
        coords = get_accurate_coords(url, driver=driver)
        results.append(coords)

        status = (f"{coords['Latitude']}, {coords['Longitude']}"
                  if coords["Latitude"] else "not found")
        print(f"[{position}/{total}] {status}")

        if ROW_DELAY and position < total:
            time.sleep(ROW_DELAY)
finally:
    driver.quit()

df_coords = pd.DataFrame(results, index=df_places.index)

In [ ]:
df = pd.concat([df_places, df_coords], axis=1)

# Empty strings rather than NaN, so unresolved rows export as blank cells
# instead of the literal text "nan".
for column in ("Latitude", "Longitude", "Source"):
    df[column] = df[column].astype("string").fillna("")

resolved = (df["Latitude"] != "").sum()
print(f"Resolved {resolved}/{len(df)} links")
df.head(10)

### How confident is each row?`pin_url` and `pin_html` are the actual marker. Anything else is a fallback wortheyeballing before the data gets used.

In [ ]:
df["Source"].value_counts(dropna=False)

In [ ]:
# Rows that need a human look: no coordinates, or resolved via a weak fallback
needs_review = df[~df["Source"].isin(["pin_url", "pin_html"])]
print(f"{len(needs_review)} row(s) to review")
needs_review[[LINK_COLUMN, "Latitude", "Longitude", "Source"]]

## 7. Export to Excel

In [ ]:
def save_df_to_excel_formatted(df, filename, sheet_name="Sheet1",
                               font_name="Calibri", font_size=11):
    """Save a DataFrame with a highlighted header row and auto-fitted columns."""
    os.makedirs(os.path.dirname(filename) or ".", exist_ok=True)

    with pd.ExcelWriter(filename, engine="xlsxwriter") as writer:
        workbook = writer.book

        header_format = workbook.add_format({
            "bold": True,
            "fg_color": "#FFE699",
            "border": 0,
            "align": "center",
            "valign": "vcenter",
            "font_name": font_name,
            "font_size": font_size,
        })
        body_format = workbook.add_format({
            "border": 0,
            "align": "center",
            "valign": "vcenter",
            "font_name": font_name,
            "font_size": font_size,
        })

        # Write the body one row down, then lay the styled header over row 0.
        df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1, header=False)
        worksheet = writer.sheets[sheet_name]

        for index, column in enumerate(df.columns):
            longest = df[column].astype(str).str.len().max()
            width = min(max(int(longest or 0), len(str(column))) + 5, 60)
            worksheet.set_column(index, index, width, body_format)
            worksheet.write(0, index, column, header_format)

        worksheet.freeze_panes(1, 0)

    print(f"Saved to {filename}")

In [ ]:
save_df_to_excel_formatted(df, OUTPUT_FILE, sheet_name=OUTPUT_SHEET)

## 8. Optional: upload the result to Google DriveDisabled by default. The folder ID is read from an environment variable - putting areal Drive ID in a committed notebook hands anyone who reads it a pointer into yourDrive.In Colab: `os.environ["GDRIVE_FOLDER_ID"] = "..."` in a cell you do not commit, oruse Colab Secrets.

In [ ]:
# !pip install -q pydrive2
#
# from pydrive2.auth import GoogleAuth
# from pydrive2.drive import GoogleDrive
# from google.colab import auth
# from oauth2client.client import GoogleCredentials
#
# folder_id = os.environ.get("GDRIVE_FOLDER_ID")
# if not folder_id:
#     raise RuntimeError("Set GDRIVE_FOLDER_ID before running this cell.")
#
# auth.authenticate_user()
# gauth = GoogleAuth()
# gauth.credentials = GoogleCredentials.get_application_default()
# drive = GoogleDrive(gauth)
#
# gfile = drive.CreateFile({
#     "title": os.path.basename(OUTPUT_FILE),
#     "parents": [{"id": folder_id}],
# })
# gfile.SetContentFile(OUTPUT_FILE)
# gfile.Upload()
# print(f"Uploaded. File ID: {gfile.get('id')}")

---## Notes**Same logic, packaged.** `src/gmaps_gps/` contains this pipeline as an importable,unit-tested module with a CLI:```bashpython -m gmaps_gps --input data/sample_tourist_places.xlsx --output output/gps.xlsx```**Before committing this notebook**, clear the outputs. A saved output cell embedsthe full rendered table - every row, link and coordinate - inside the `.ipynb` file,where it survives long after it has scrolled off your screen:```bashjupyter nbconvert --clear-output --inplace notebooks/*.ipynb```The repo's pre-commit hook does this automatically.